In [ ]:
!pip install ijson==3.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

# Use force_remount=True to ensure the drive is remounted even if it's already mounted.
# Use timeout_ms to adjust the timeout period if your network is slow. For example, increasing it to 300000 for a 5-minute timeout.
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)

Mounted at /content/drive


In [ ]:
import ijson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from glob import glob

# loading in raw xsens and insole data

In [ ]:
import pandas as pd
import os

def load_files_to_separate_dfs(main_folder_path):
    xsens_dfs = []
    insole_dfs = []

    for root, dirs, files in os.walk(main_folder_path):
        if 'xsens.csv' in files and 'insoles.csv' in files and 'labels.csv' in files:
            # Load the individual files
            df_xsens = pd.read_csv(os.path.join(root, 'xsens.csv'))
            df_insole = pd.read_csv(os.path.join(root, 'insoles.csv'))
            df_labels = pd.read_csv(os.path.join(root, 'labels.csv'))

            # Merge the labels with each type of data
            df_xsens = pd.merge(df_xsens, df_labels, on=['time', 'participant_id', 'task'])
            df_insole = pd.merge(df_insole, df_labels, on=['time', 'participant_id', 'task'])

            # Extract participant number from the folder path
            participant_num = os.path.basename(root)
            df_xsens['participant_num'] = participant_num
            df_insole['participant_num'] = participant_num

            # Append to respective lists
            xsens_dfs.append(df_xsens)
            insole_dfs.append(df_insole)

            print(f"Successfully loaded data from folder: {participant_num}, {root}")

    return xsens_dfs, insole_dfs


In [ ]:
# Usage example
main_folder_path = '/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set'
xsens_dataframes, insole_dataframes = load_files_to_separate_dfs(main_folder_path)


Successfully loaded data from folder: id15, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id15
Successfully loaded data from folder: id01, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id01
Successfully loaded data from folder: id12, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id12
Successfully loaded data from folder: id14, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id14
Successfully loaded data from folder: id24, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id24
Successfully loaded data from folder: id23, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id23
Successfully loaded data from folder: id08, /content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB/id08
Successfully loaded data from folder: id07, /content/drive/MyDrive/Next_Step

In [ ]:
# len(xsens_dataframes)

In [ ]:
def merge_by_participant(honda_dfs):
    merged_dfs = {}
    for df in honda_dfs:
        # Ensure the DataFrame contains 'participant_id' column
        if 'participant_id' not in df.columns:
            raise ValueError("The DataFrame is missing the 'participant_id' column.")

        # Get the participant ID
        participant_id = df['participant_id'].unique()
        if len(participant_id) != 1:
            raise ValueError(f"DataFrame contains multiple or no participant IDs: {participant_id}")

        participant_id = participant_id[0]  # Extract the single participant ID

        # Merge the DataFrame for the participant
        if participant_id not in merged_dfs:
            merged_dfs[participant_id] = df
        else:
            merged_dfs[participant_id] = pd.concat([merged_dfs[participant_id], df], ignore_index=True)

    # Return a list of DataFrames
    return list(merged_dfs.values())

In [ ]:
merged_imu_dataframes = merge_by_participant(xsens_dataframes)
merged_insole_dataframes = merge_by_participant(insole_dataframes)

In [ ]:
all_sensor_types = ['orientation','position','velocity','acceleration','angularVelocity','angularAcceleration','sensorFreeAcceleration','sensorMagneticField','sensorOrientation','jointAngle','jointAngleXZY', 'Left', 'Right', 'Right__raw', 'Left__raw', 'Left__norm', 'Right__norm']

all_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe', 'Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes']

imu_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

insole_sensor_locations = ['Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes', "RightFoot", "LeftFoot"]

axes = ['x', 'y', 'z', 'ql', 'qi', 'qj', 'qk', 'norm', 'raw']

all_surface_types = ['walk', 'slope_down', 'slope_up', 'stairs_up', 'stairs_down', 'pavement_down', 'pavement_up']

all_sensor_types = ['orientation','position','velocity','acceleration','angularVelocity','angularAcceleration','sensorFreeAcceleration','sensorMagneticField','sensorOrientation','jointAngle','jointAngleXZY', 'Left', 'Right', 'Acc', 'Gyr', "Mag"]

columns = ['walk_mode', 'time', 'participant_id', 'task', 'sensor_location', 'orientation_q1', 'orientation_qi', 'orientation_qj', 'orientation_qk',
           'position_x', 'position_y', 'position_z', 'velocity_x', 'velocity_y', 'velocity_z', 'acceleration_x', 'acceleration_y', 'acceleration_z',
           'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z', 'angularAcceleration_x', 'angularAcceleration_y', 'angularAcceleration_z',
           'sensorFreeAcceleration_x', 'sensorFreeAcceleration_y', 'sensorFreeAcceleration_z', 'sensorMagneticField_x', 'sensorMagneticField_y',
           'sensorMagneticField_z', 'sensorOrientation_q1', 'sensorOrientation_qi', 'sensorOrientation_qj', 'sensorOrientation_qk', 'jointAngle_x',
           'jointAngle_y', 'jointAngle_z', 'jointAngleXZY_x', 'jointAngleXZY_y', 'jointAngleXZY_z', 'Left', 'Right', 'Right__raw', 'Left__raw', 'Left__norm', 'Right__norm']


In [ ]:
def process_trial_df(trial_df):
    df_list = []
    print(f"Columns in the DataFrame: {list(trial_df.columns)}")

    # Filter sensor locations that are present in the DataFrame columns
    valid_sensor_locations = [
        loc for loc in all_sensor_locations if any(f"_{loc}_" in col for col in trial_df.columns)
    ]

    # Process each valid sensor location
    for sensor_location in valid_sensor_locations:
        df_by_sensor = trial_df[['time', 'participant_id', 'task', 'walk_mode', 'insoles_RightFoot_is_step', 'insoles_RightFoot_is_lifted', 'insoles_LeftFoot_is_step', 'insoles_LeftFoot_is_lifted']].copy()
        df_by_sensor['sensor_location'] = sensor_location

        # Iterate through all sensor types and axes
        for sensor_type in all_sensor_types:
            for axis in axes:
                original_col_name = f"{sensor_type}_{sensor_location}_{axis}"

                if original_col_name in trial_df.columns:
                    # Create new column for sensor data
                    new_col_name = f"{sensor_type}_{axis}"
                    df_by_sensor[new_col_name] = trial_df[original_col_name]

        # Append only if there is relevant data
        if len(df_by_sensor.columns) > 5:  # Metadata + at least one data column
            df_list.append(df_by_sensor)
        else:
            print(f"No data found for sensor location: {sensor_location}")

    return df_list

In [ ]:
# Loop through dataframes
count = 0
total = len(xsens_dataframes)
for df in xsens_dataframes:
    count += 1
    print(f"Processing {count}/{total} dataframes")
    processed_dfs = process_trial_df(df)

    for processed_df in processed_dfs:
        participant_id = processed_df['participant_id'].unique()[0]
        sensor_location = processed_df['sensor_location'].unique()[0]
        task = processed_df['task'].unique()[0]

        # Save the processed DataFrame to a CSV
        output_path = f"/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/processed_data/participant_{participant_id}_{sensor_location}_{task}.csv"
        processed_df.to_csv(output_path, index=False)
        print(f"Saved: {output_path}, Shape: {processed_df.shape}")

Processing 1/60 dataframes
Columns in the DataFrame: ['time', 'participant_id', 'task', 'orientation_Pelvis_q1', 'orientation_Pelvis_qi', 'orientation_Pelvis_qj', 'orientation_Pelvis_qk', 'orientation_L5_q1', 'orientation_L5_qi', 'orientation_L5_qj', 'orientation_L5_qk', 'orientation_L3_q1', 'orientation_L3_qi', 'orientation_L3_qj', 'orientation_L3_qk', 'orientation_T12_q1', 'orientation_T12_qi', 'orientation_T12_qj', 'orientation_T12_qk', 'orientation_T8_q1', 'orientation_T8_qi', 'orientation_T8_qj', 'orientation_T8_qk', 'orientation_Neck_q1', 'orientation_Neck_qi', 'orientation_Neck_qj', 'orientation_Neck_qk', 'orientation_Head_q1', 'orientation_Head_qi', 'orientation_Head_qj', 'orientation_Head_qk', 'orientation_RightShoulder_q1', 'orientation_RightShoulder_qi', 'orientation_RightShoulder_qj', 'orientation_RightShoulder_qk', 'orientation_RightUpperArm_q1', 'orientation_RightUpperArm_qi', 'orientation_RightUpperArm_qj', 'orientation_RightUpperArm_qk', 'orientation_RightForeArm_q1', '

In [ ]:
# insole_dataframes[0]

In [ ]:
import pandas as pd
import glob

def process_trial_df(trial_df):

    df_list = []
    print(f"Columns in the DataFrame: {list(trial_df.columns)}")

    # Filter sensor locations that are present in the DataFrame columns
    valid_sensor_locations = [
        loc for loc in all_sensor_locations if any(f"{loc}_" in col or f"_{loc}_" in col or f"_{loc}" in col for col in trial_df.columns)
    ]
    print(valid_sensor_locations)
    # Process each valid sensor location
    for sensor_location in valid_sensor_locations:
        df_by_sensor = trial_df[['time', 'participant_id', 'task', 'walk_mode', 'insoles_RightFoot_is_step', 'insoles_RightFoot_is_lifted', 'insoles_LeftFoot_is_step', 'insoles_LeftFoot_is_lifted']].copy()
        df_by_sensor['sensor_location'] = sensor_location

        # Iterate through all sensor types and axes
        for sensor_type in all_sensor_types:
            for axis in axes:
                original_col_name = f"{sensor_type}_{sensor_location}_{axis}"

                if original_col_name in trial_df.columns:
                    # Create new column for sensor data
                    new_col_name = f"{sensor_type}_{axis}"
                    df_by_sensor[new_col_name] = trial_df[original_col_name]

        # Append only if there is relevant data
        if len(df_by_sensor.columns) > 5:  # Metadata + at least one data column
            df_list.append(df_by_sensor)
        else:
            print(f"No data found for sensor location: {sensor_location}")

    return df_list

In [ ]:
# Loop through dataframes
count = 0
total = len(insole_dataframes)
for df in insole_dataframes:
    count += 1
    print(f"Processing {count}/{total} dataframes")
    processed_dfs = process_trial_df(df)

    for processed_df in processed_dfs:
        participant_id = processed_df['participant_id'].unique()[0]
        sensor_location = processed_df['sensor_location'].unique()[0]
        task = processed_df['task'].unique()[0]

        # Save the processed DataFrame to a CSV
        output_path = f"/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/processed_data/participant_{participant_id}_{sensor_location}_{task}.csv"
        processed_df.to_csv(output_path, index=False)
        print(f"Saved: {output_path}, Shape: {processed_df.shape}")

Processing 1/60 dataframes
Columns in the DataFrame: ['Unnamed: 0', 'time', 'participant_id', 'task', 'Left_Hallux', 'Left_Toes', 'Left_Met1', 'Left_Met3', 'Left_Met5', 'Left_Arch', 'Left_Heel_R', 'Left_Heel_L', 'Left_Acc_x', 'Left_Acc_y', 'Left_Acc_z', 'Left_Gyr_x', 'Left_Gyr_y', 'Left_Gyr_z', 'Left_Mag_x', 'Left_Mag_y', 'Left_Mag_z', 'Left_Temp', 'Right_Hallux', 'Right_Toes', 'Right_Met1', 'Right_Met3', 'Right_Met5', 'Right_Arch', 'Right_Heel_L', 'Right_Heel_R', 'Right_Acc_x', 'Right_Acc_y', 'Right_Acc_z', 'Right_Gyr_x', 'Right_Gyr_y', 'Right_Gyr_z', 'Right_Mag_x', 'Right_Mag_y', 'Right_Mag_z', 'Right_Temp', 'Right_Toes_raw', 'Right_Hallux_raw', 'Right_Met5_raw', 'Right_Met3_raw', 'Right_Met1_raw', 'Right_Arch_raw', 'Right_Heel_R_raw', 'Right_Heel_L_raw', 'Left_Toes_raw', 'Left_Hallux_raw', 'Left_Met5_raw', 'Left_Met3_raw', 'Left_Met1_raw', 'Left_Arch_raw', 'Left_Heel_R_raw', 'Left_Heel_L_raw', 'Right_Acc_x_raw', 'Right_Acc_y_raw', 'Right_Acc_z_raw', 'Right_Mag_x_raw', 'Right_Mag_y_r